# Fit a regular vine

This example loads six asset price series from `prices.csv`, fits a six-node vine, and reads its final conditional probabilities.

In [1]:
from helpers import ASSET_NAMES, PRICES_PATH, load_assets
from vines import Method, Vine

In [2]:
# Load asset price data
assets = load_assets()
assert len(assets) == 6
print(f"loaded {len(assets[0].prices())} prices for {', '.join(ASSET_NAMES)} from {PRICES_PATH}")

loaded 1999 prices for BTC, ETH, LINK, XRP, SOL, DOGE from /Users/shaun/Code/DEVELOPMENT/cpp-vines/prices.csv


In [3]:
# Determine best fitting R-Vine copula
marginals = [asset.u_values() for asset in assets]
vine = Vine(marginals, max_nodes=6, method=Method.CMPI)
vine.fit()
print(f"{vine}")

T1: 12, 23, 25, 16, 24
T2: 34|2, 26|1, 14|2, 35|2
T3: 46|12, 45|23, 13|24
T4: 36|124, 15|234
T5: 56|1234


In [4]:
# Print the latest results
results = vine.final_results()
print(f"latest left given right: {results.left_given_right[-1]:.2f}")
print(f"latest right given left: {results.right_given_left[-1]:.2f}")

latest left given right: -36.90
latest right given left: -19.96


In [5]:
from statistics import fmean, stdev

import matplotlib.pyplot as plt

cmpi = results.left_given_right[-1000:]
windows = [cmpi[i:i + 20] for i in range(len(cmpi) - 19)]
middle = [fmean(window) for window in windows]
width = [2 * stdev(window) for window in windows]
x = range(19, len(cmpi))

plt.plot(cmpi, label="CMPI")
plt.plot(x, middle, label="20-period mean")
plt.fill_between(x, [m - w for m, w in zip(middle, width)], [m + w for m, w in zip(middle, width)], alpha=0.2, label="Bollinger band")
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'